<a href="https://colab.research.google.com/github/Ferasman979/OptiMulti-Video/blob/dev/notebooks/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiMulti-Video: High-Performance Multimodal Attention (T4 Gpus)

This notebook demonstrates the **OptiMulti-Video** project, featuring:
1. **Custom CUDA Kernel**: Fused Normalization & Projection.
2. **Distributed Training**: FSDP on T4 GPUs.

## 1. Environment Setup

In [1]:
!nvidia-smi

Wed Jan 28 00:40:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the Code
Running this on Colab requires the source code.
**Option A (Recommended)**: Clone your GitHub repository.
**Option B**: Upload the `src/`, `model/`, `training/` folders and `setup.py` manually to the Files tab.

In [2]:
!rm -rf OptiMulti-Video
!git clone https://github.com/Ferasman979/OptiMulti-Video.git
%cd OptiMulti-Video

Cloning into 'OptiMulti-Video'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 30 (delta 7), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 14.94 KiB | 2.49 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/OptiMulti-Video


## 3. Compile Custom CUDA Kernel
We use `pip install .` to compile the C++ extension on the attached GPU.

In [3]:
!pip install -v .

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing /content/OptiMulti-Video
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info
  writing /tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/SOURCES.txt'
  W0128 00:41:01.886000 4508 torch/utils/cpp_extension.py:630] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
  reading manifest file '/tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-py0df16t/optimulti_fusion.egg-info/SO

## 4. Run FSDP Distributed Training
We spawn 2 processes (if 2 GPUs are available) to train the model.

In [4]:
!python training/train_fsdp.py

[OptiMulti] CUDA Extension Loaded Successfully.
Spawning 1 processes for FSDP training...
[OptiMulti] CUDA Extension Loaded Successfully.
/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/_init_utils.py:430: UserWarning: FSDP is switching to use `NO_SHARD` instead of ShardingStrategy.FULL_SHARD since the world size is 1.
  warnings.warn(
[Rank 0] Addr: FullyShardedDataParallel(
  (_fsdp_wrapped_module): OptiMultiVideo(
    (vision_encoder): VisionEncoder(
      (patch_embed): Sequential(
        (0): Conv3d(3, 768, kernel_size=(1, 32, 32), stride=(1, 32, 32))
        (1): Flatten(start_dim=2, end_dim=-1)
      )
    )
    (text_decoder): TextDecoder(
      (embedding): Embedding(1000, 768)
      (pos_emb): Embedding(64, 768)
      (decoder_layer): TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (multihead_attn): MultiheadAttention(
       

## 5. Verify Custom Kernel
Let's run a quick numerical check to ensure our CUDA kernel matches PyTorch.

In [5]:
import torch
import optimulti_fusion_cuda

if torch.cuda.is_available():
    device = torch.device('cuda')
    a = torch.randn(16, 128, 768, device=device)
    b = torch.randn(16, 128, 768, device=device)
    out_cuda = torch.zeros_like(a)

    # Custom Op
    optimulti_fusion_cuda.fused_add_layernorm(a, b, out_cuda, 1e-5)

    # PyTorch Ref
    out_ref = torch.nn.functional.layer_norm(a + b, (768,), eps=1e-5)

    diff = (out_cuda - out_ref).abs().max().item()
    print(f"Max Difference: {diff}")
    assert diff < 1e-3, "Kernel mismatch!"
    print("verification Passed!")
else:
    print("No GPU available for verification.")

Max Difference: 9.5367431640625e-07
verification Passed!
